# LLM Enrichment for Assembled EMV Sections

This notebook takes your final structured JSON and adds an **LLM enrichment layer** before chunking and embeddings.

It does **not** redo layout extraction or heading detection.  
It assumes you already have a grounded structure like:

- `section_id`
- `parent_section_id`
- `title`
- `level`
- `parent_titles`
- `blocks`
- `tables`

## What this notebook does

For each section, it uses an LLM to:

1. **Clean and normalize text**
2. **Semantically label blocks**  
   Examples: `definition`, `requirement`, `note`, `warning`, `procedure`, `description`
3. **Add metadata enrichment**
   - short summary
   - keywords
   - topics
4. **Suggest chunking actions**
   - merge small related blocks
   - split very large content at semantic boundaries
5. **Optionally repair table structure**
6. **Optionally resolve explicit cross-references**

## Input file

This notebook is configured to read:

```python
..\..\..\..\assembled\cleaned_assembled_sections.json
```

## Output files

It writes:

- `enriched_sections.json`
- `chunk_ready_sections.json`

The notebook is written to be easy to adapt for:
- **Ollama** local models
- or another OpenAI-compatible endpoint


## 1) Configuration

Update the configuration below if needed.

By default, this notebook is set up for **Ollama** using a local model such as:

- `mistral`
- `llama3`
- `qwen2.5`

You can change the model name later without changing the rest of the notebook.


In [ ]:
from pathlib import Path
import json
import re
import time
from typing import Any, Dict, List, Optional
import requests

# ---------- Paths ----------
INPUT_JSON = Path(r"..\..\..\..\assembled\cleaned_assembled_sections.json")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_JSON = OUTPUT_DIR / "enriched_sections.json"
CHUNK_READY_JSON = OUTPUT_DIR / "chunk_ready_sections.json"

# ---------- LLM ----------
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "mistral"   # change if needed

# ---------- Processing ----------
MAX_SECTIONS = None      # set to an integer for testing, e.g. 5
SLEEP_BETWEEN_CALLS = 0.2
RETRY_COUNT = 2
TIMEOUT_SECONDS = 180

# ---------- Behavior ----------
ENABLE_TABLE_REPAIR = True
ENABLE_CROSS_REFERENCES = True
SAVE_EVERY_N = 10

print("Input:", INPUT_JSON)
print("Output directory:", OUTPUT_DIR.resolve())


## 2) Load the assembled JSON

This reads your final manually assembled structure.  
Each entry is expected to be a section object with nested `blocks` and `tables`.


In [ ]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    sections = json.load(f)

if MAX_SECTIONS is not None:
    sections = sections[:MAX_SECTIONS]

print(f"Loaded {len(sections)} sections.")
print("Example keys:", list(sections[0].keys()) if sections else "No data")


## 3) Prompt design

This is the main LLM prompt.

The prompt tells the model **not** to invent structure and **not** to rewrite technical meaning.  
It only performs enrichment on top of your grounded JSON.

### What the prompt asks for

For each section, the model should:

- clean paragraph text while preserving meaning
- classify each block semantically
- summarize the section
- extract keywords and topics
- propose chunking groups
- repair tables when possible
- resolve explicit references such as `See Section 6.5`

The model must return **strict JSON only**.


In [ ]:
ENRICHMENT_PROMPT = """You are enriching a technically grounded JSON extracted from EMV specification documents.

Your job is NOT to re-parse the PDF and NOT to invent missing structure.
You must preserve the original hierarchy, pages, and ids.
You are only allowed to clean, label, enrich, and prepare the content for chunking.

INPUT:
You will receive one section object in JSON format.
It already contains:
- section metadata
- blocks with page numbers and bounding boxes
- tables if available

TASKS:
1) Text cleaning and normalization
- Clean the text of each paragraph-like block
- Merge broken line artifacts
- Normalize spacing and punctuation
- Remove obvious extraction noise only if it is clearly noise
- Preserve all technical terms, acronyms, values, and requirements exactly
- Do NOT summarize in place
- Do NOT remove important normative language such as must, shall, should, may

2) Semantic block labeling
For each block, assign one label from:
- heading
- definition
- requirement
- note
- warning
- procedure
- description
- list
- caption
- reference
- formula
- table_intro
- other

Rules:
- Use requirement for normative statements and explicit obligations
- Use definition for definitional text
- Use procedure for ordered or action-driven steps
- Use note for explanatory or supplementary text
- Use warning for cautionary content
- Use description for neutral explanatory technical text
- Use reference for cross-references or bibliography-like mentions

3) Metadata enrichment at section level
Generate:
- section_summary: 1 to 3 sentences, factual, no hallucinations
- keywords: 5 to 12 high-value search keywords
- topics: 2 to 6 broader themes

4) Chunking preparation
Create chunking recommendations that prepare the section for downstream chunk creation.
You must:
- keep semantically atomic units intact when possible
- avoid splitting definitions and short requirements unnecessarily
- recommend splitting only when content is too long or covers multiple subtopics
- recommend merging tiny adjacent blocks only if they belong to the same idea

Return:
- recommended_chunk_plan: a list of chunk groups
Each chunk group must contain:
  - chunk_label
  - rationale
  - block_ids
  - estimated_cohesion: high / medium / low

5) Table repair (if tables exist)
For each table:
- preserve as much raw content as possible
- identify likely headers when clear
- produce a normalized representation:
  - table_id
  - title (if inferable from nearby text, otherwise null)
  - headers
  - rows
  - notes
If the table is too ambiguous, preserve it with a warning instead of inventing structure.

6) Cross-reference resolution
If the section explicitly mentions references like:
- See Section 6.5
- Refer to Book 2
extract them into:
- cross_references: list of objects with:
  - raw_text
  - reference_type: intra_section / intra_book / cross_book / external
  - target_hint
Do not invent targets that are not explicit.

OUTPUT FORMAT:
Return valid JSON only.
Use exactly this top-level structure:

{
  "section_id": "...",
  "cleaned_blocks": [
    {
      "block_id": "...",
      "clean_text": "...",
      "semantic_type": "...",
      "quality_notes": []
    }
  ],
  "section_enrichment": {
    "section_summary": "...",
    "keywords": ["..."],
    "topics": ["..."]
  },
  "recommended_chunk_plan": [
    {
      "chunk_label": "...",
      "rationale": "...",
      "block_ids": ["..."],
      "estimated_cohesion": "high"
    }
  ],
  "normalized_tables": [
    {
      "table_id": "...",
      "title": null,
      "headers": [],
      "rows": [],
      "notes": []
    }
  ],
  "cross_references": [
    {
      "raw_text": "...",
      "reference_type": "...",
      "target_hint": "..."
    }
  ]
}

STRICT RULES:
- Output JSON only
- Do not wrap in markdown
- Do not add commentary
- Do not drop block ids
- Do not alter source metadata not requested here
- If information is uncertain, preserve ambiguity and mention it in notes
- Prefer faithful preservation over aggressive cleanup
"""
print(ENRICHMENT_PROMPT[:1200])

## 4) LLM call helper

This helper sends one section at a time to an Ollama model and expects JSON back.

If your model sometimes adds extra text, the parsing helper in the next cell will try to recover the JSON object.


In [ ]:
def call_ollama_json(prompt: str, model: str = MODEL_NAME) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": "json"
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    data = response.json()
    return data["response"]


## 5) Robust JSON extraction

Some local models still produce extra tokens around JSON.  
This helper tries a normal parse first, then falls back to extracting the largest JSON object.


In [ ]:
def parse_json_response(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        return json.loads(candidate)

    raise ValueError("Could not parse valid JSON from model response.")


## 6) Section packaging

We keep the original section object intact and only send the fields needed for enrichment.
This keeps the prompt smaller and more stable.


In [ ]:
def build_section_payload(section: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "section_id": section.get("section_id"),
        "parent_section_id": section.get("parent_section_id"),
        "doc_id": section.get("doc_id"),
        "doc_version": section.get("doc_version"),
        "doc_date": section.get("doc_date"),
        "title": section.get("title"),
        "section_number": section.get("section_number"),
        "level": section.get("level"),
        "parent_titles": section.get("parent_titles", []),
        "start_page": section.get("start_page"),
        "end_page": section.get("end_page"),
        "blocks": section.get("blocks", []),
        "tables": section.get("tables", []) if ENABLE_TABLE_REPAIR else []
    }

sample_payload = build_section_payload(sections[0]) if sections else {}
print(json.dumps(sample_payload, indent=2)[:2500])


## 7) Enrichment function

This function:
- builds the prompt
- calls the model
- parses the JSON output
- retries on failure


In [ ]:
def enrich_one_section(section: Dict[str, Any]) -> Dict[str, Any]:
    payload = build_section_payload(section)
    full_prompt = ENRICHMENT_PROMPT + "\n\nSECTION_JSON:\n" + json.dumps(payload, ensure_ascii=False)

    last_error = None
    for attempt in range(RETRY_COUNT + 1):
        try:
            raw = call_ollama_json(full_prompt)
            enriched = parse_json_response(raw)
            return enriched
        except Exception as e:
            last_error = e
            if attempt < RETRY_COUNT:
                time.sleep(1.2 * (attempt + 1))
            else:
                raise last_error


## 8) Run enrichment on all sections

This may take time depending on:
- number of sections
- model size
- CPU/GPU availability

Progress is checkpointed every `SAVE_EVERY_N` sections.


In [ ]:
enriched_outputs: List[Dict[str, Any]] = []

for i, section in enumerate(sections, start=1):
    enriched = enrich_one_section(section)
    enriched_outputs.append(enriched)

    if i % SAVE_EVERY_N == 0 or i == len(sections):
        with open(ENRICHED_JSON, "w", encoding="utf-8") as f:
            json.dump(enriched_outputs, f, ensure_ascii=False, indent=2)

    print(f"[{i}/{len(sections)}] Enriched {section.get('section_id')}")
    time.sleep(SLEEP_BETWEEN_CALLS)

print("Saved:", ENRICHED_JSON)


## 9) Merge enriched output back into the original sections

This step joins:
- original grounded section data
- LLM-enriched block and section metadata

The result is still section-centric and easier to inspect before final chunk creation.


In [ ]:
def index_by_section_id(items: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {item["section_id"]: item for item in items if "section_id" in item}

enriched_index = index_by_section_id(enriched_outputs)

merged_sections = []
for section in sections:
    sid = section["section_id"]
    enrich = enriched_index.get(sid, {})

    cleaned_map = {
        b["block_id"]: b
        for b in enrich.get("cleaned_blocks", [])
        if "block_id" in b
    }

    new_blocks = []
    for block in section.get("blocks", []):
        b2 = dict(block)
        extra = cleaned_map.get(block.get("block_id"), {})
        b2["clean_text"] = extra.get("clean_text", block.get("text"))
        b2["semantic_type"] = extra.get("semantic_type", "other")
        b2["quality_notes"] = extra.get("quality_notes", [])
        new_blocks.append(b2)

    s2 = dict(section)
    s2["blocks"] = new_blocks
    s2["section_summary"] = enrich.get("section_enrichment", {}).get("section_summary")
    s2["keywords"] = enrich.get("section_enrichment", {}).get("keywords", [])
    s2["topics"] = enrich.get("section_enrichment", {}).get("topics", [])
    s2["recommended_chunk_plan"] = enrich.get("recommended_chunk_plan", [])
    s2["normalized_tables"] = enrich.get("normalized_tables", [])
    s2["cross_references"] = enrich.get("cross_references", []) if ENABLE_CROSS_REFERENCES else []

    merged_sections.append(s2)

len(merged_sections)


## 10) Build chunk-ready records

This notebook does not force one universal chunk size.  
Instead, it uses the LLM's `recommended_chunk_plan` to create semantically coherent candidate chunks.

### Strategy

For each recommended chunk group:
- collect the grouped block texts
- preserve section metadata
- preserve source pages
- keep block ids for traceability

You can later add token-based splitting if a chunk is still too large.


In [ ]:
def build_chunk_text_from_blocks(blocks: List[Dict[str, Any]]) -> str:
    parts = []
    for b in blocks:
        txt = b.get("clean_text") or b.get("text") or ""
        txt = txt.strip()
        if txt:
            parts.append(txt)
    return "\n\n".join(parts).strip()

chunk_ready_records = []

for section in merged_sections:
    block_map = {b["block_id"]: b for b in section.get("blocks", []) if "block_id" in b}
    chunk_plan = section.get("recommended_chunk_plan", [])

    if not chunk_plan:
        # fallback: one section-level chunk
        block_ids = [b["block_id"] for b in section.get("blocks", []) if "block_id" in b]
        selected_blocks = [block_map[bid] for bid in block_ids if bid in block_map]
        chunk_ready_records.append({
            "chunk_id": f"{section['section_id']}__chunk_1",
            "section_id": section["section_id"],
            "doc_id": section.get("doc_id"),
            "doc_version": section.get("doc_version"),
            "doc_date": section.get("doc_date"),
            "title": section.get("title"),
            "section_number": section.get("section_number"),
            "level": section.get("level"),
            "parent_titles": section.get("parent_titles", []),
            "start_page": section.get("start_page"),
            "end_page": section.get("end_page"),
            "chunk_label": "fallback_full_section",
            "chunk_rationale": "No chunk plan was returned, so the whole section was preserved.",
            "block_ids": block_ids,
            "text": build_chunk_text_from_blocks(selected_blocks),
            "keywords": section.get("keywords", []),
            "topics": section.get("topics", []),
            "section_summary": section.get("section_summary"),
            "cross_references": section.get("cross_references", [])
        })
        continue

    for idx, grp in enumerate(chunk_plan, start=1):
        block_ids = grp.get("block_ids", [])
        selected_blocks = [block_map[bid] for bid in block_ids if bid in block_map]
        text = build_chunk_text_from_blocks(selected_blocks)

        chunk_ready_records.append({
            "chunk_id": f"{section['section_id']}__chunk_{idx}",
            "section_id": section["section_id"],
            "doc_id": section.get("doc_id"),
            "doc_version": section.get("doc_version"),
            "doc_date": section.get("doc_date"),
            "title": section.get("title"),
            "section_number": section.get("section_number"),
            "level": section.get("level"),
            "parent_titles": section.get("parent_titles", []),
            "start_page": section.get("start_page"),
            "end_page": section.get("end_page"),
            "chunk_label": grp.get("chunk_label"),
            "chunk_rationale": grp.get("rationale"),
            "estimated_cohesion": grp.get("estimated_cohesion"),
            "block_ids": block_ids,
            "text": text,
            "keywords": section.get("keywords", []),
            "topics": section.get("topics", []),
            "section_summary": section.get("section_summary"),
            "cross_references": section.get("cross_references", [])
        })

print(f"Built {len(chunk_ready_records)} chunk-ready records.")


## 11) Save outputs

We save both:
- the **section-enriched** version
- the **chunk-ready** version


In [ ]:
with open(ENRICHED_JSON, "w", encoding="utf-8") as f:
    json.dump(merged_sections, f, ensure_ascii=False, indent=2)

with open(CHUNK_READY_JSON, "w", encoding="utf-8") as f:
    json.dump(chunk_ready_records, f, ensure_ascii=False, indent=2)

print("Saved enriched sections to:", ENRICHED_JSON)
print("Saved chunk-ready records to:", CHUNK_READY_JSON)


## 12) Quick inspection

Check a few results before embedding.

Things to verify:
- cleaned text did not change technical meaning
- semantic labels make sense
- chunk groups are coherent
- keywords/topics are useful
- tables are not hallucinated


In [ ]:
# Inspect one enriched section
example_section = merged_sections[0]
print(json.dumps(example_section, indent=2, ensure_ascii=False)[:5000])


In [ ]:
# Inspect one chunk-ready record
example_chunk = chunk_ready_records[0]
print(json.dumps(example_chunk, indent=2, ensure_ascii=False)[:3000])


## 13) Notes and practical advice

### Best first run
Start with:
- `MAX_SECTIONS = 3`
- inspect outputs manually
- then scale up

### Recommended model behavior
A strong local instruct model usually works better than a tiny model.
For example:
- Mistral Instruct
- Llama 3 Instruct
- Qwen Instruct

### Important
This notebook is designed for **enrichment**, not authority replacement.
Your grounded JSON remains the source of truth.

### Next possible step
After this, you can add:
- token counting
- hard split of overlong chunks
- embeddings
- Chroma ingestion
